# TBATS User Guide

TBATS is a flexible time-series forecasting model designed to handle complex seasonality, nonlinear trends, and non-Gaussian data. The name stands for:

- **T**rigonometric seasonality
- **B**ox–Cox transformation
- **A**RMA errors
- **T**rend
- **S**easonal components

This implementation provides both:
- **TBATS**: fixed configuration
- **AutoTBATS**: automatic model selection

In [ ]:
import sys
import os

# Add parent directory to path so we can import the library modules
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import jax
from jax import config
config.update("jax_enable_x64", True)

import jax.numpy as jnp
import numpy as np

from tbats_model import AutoTBATS, TBATS
from conformal_intervals import ConformalIntervals

# Math Overview

TBATS decomposes the series into:

1. A **level** and **trend** component (with optional damping)
2. One or more **seasonal** components, represented using Fourier (trigonometric) terms
3. **ARMA** structure on the residuals to capture remaining autocorrelation
4. **Box–Cox** transformation to stabilize variance

The key innovation is using trigonometric representations for seasonality, which allows the model to handle multiple and non-integer seasonal periods efficiently within a single state-space framework.

# When to Use

**Use TBATS when your data exhibits:**
1. **Multiple or non-integer seasonal periods** (e.g. daily + weekly, or 365.25-day annual cycles)
2. **Long seasonal cycles** that would require many states in a standard ETS model
3. **Changing seasonal patterns** that evolve over time
4. **Non-constant variance** (Box–Cox helps stabilize it) — in contrast, ARIMA assumes constant error variance

**Avoid when:**
1. Seasonality is simple and well-defined — ETS or seasonal ARIMA are faster and simpler
2. Data is short relative to the seasonal period (TBATS needs at least 2–3 full cycles)

# API Contract

Our TBATS implementation inherits AutoTBATS, which automatically selects the following parameters:
* Trend
* Damped trend
* Box–Cox usage
* ARMA errors

## Conformal Intervals in TBATS

TBATS supports both:
1. **Model-native (Gaussian)** prediction intervals
2. **Conformal** prediction intervals

When `conformal_params` is provided, conformal intervals are added on top of TBATS forecasts.

## Constructor

```python
AutoTBATS(
    season_length: Union[int, List[int]],
    use_boxcox: Optional[bool] = None,
    bc_lower_bound: float = -1.0,
    bc_upper_bound: float = 2.0,
    use_trend: Optional[bool] = None,
    use_damped_trend: Optional[bool] = None,
    use_arma_errors: bool = False,
    alias: str = "AutoTBATS",
    conformal_params: Optional[ConformalIntervals] = None,
)
```

**Key Parameters:**
- `season_length`: One or more seasonal periods (e.g. `[24, 168]` for hourly + weekly)
- `use_boxcox`: Whether to apply a Box–Cox transformation
- `use_trend`, `use_damped_trend`: Controls inclusion of trend components
- `use_arma_errors`: Whether residuals follow an ARMA process
- `conformal_params`: Enables conformal prediction intervals

## Methods

### Fit
```python
fit(y: ArrayLike, X: Optional[ArrayLike] = None) -> AutoTBATS
```
Fits TBATS with automatic component selection. Applies Box–Cox if enabled (requires strictly positive data).

### Predict
```python
predict(h: int, level: Optional[List[int]] = None) -> Dict[str, ArrayLike]
```
Returns `"mean"` (h,) point forecasts and optionally `"lo-{level}"`, `"hi-{level}"` prediction intervals.

### Predict in Sample
```python
predict_in_sample(level: Optional[List[int]] = None) -> Dict[str, ArrayLike]
```
Returns `"fitted"` in-sample fitted values. Useful for diagnostics and residual analysis.

### Forecast (Stateless)
```python
forecast(y, h, level=None, fitted=False) -> Dict[str, ArrayLike]
```
Stateless fit + predict. Returns `"mean"`, optionally `"fitted"`, and prediction intervals.

# Examples

## Basic Fit + Predict

In [ ]:
# Basic fit and predict workflow
print("[Test 1] Basic fit and predict")
y = jnp.array([10., 12., 13., 15., 17., 20., 22., 25., 27., 30.])

model = AutoTBATS(
    season_length=3,
    use_boxcox=False,
    use_trend=True,
    use_damped_trend=False,
    use_arma_errors=False
)
model.fit(y)

result = model.predict(h=3, level=None)

print("Forecast keys:", list(result.keys()))
print("Forecast:", result["mean"])
print("Shape:", result["mean"].shape)

assert "mean" in result
assert result["mean"].shape == (3,)
assert not jnp.any(jnp.isnan(result["mean"]))

print("Basic fit and predict works!")

This example demonstrates the standard **stateful** workflow for TBATS:

1. **Fit** the model once on training data
2. **Predict** future values from the fitted state

The model is configured with a single seasonal period (`season_length=3`), no Box–Cox transformation, a deterministic trend, and no ARMA errors (keeping dynamics simple). The output contains only a `"mean"` key since no intervals were requested.

## Basic Forecast (Stateless)

In [ ]:
# Stateless forecast with fitted values
print("[Test 2] Forecast with fitted values")
y = jnp.array([5., 7., 6., 8., 10., 9., 11., 13.])

model = TBATS(season_length=4, use_boxcox=False)

result = model.forecast(y=y, h=2, fitted=True, level=None)

print("Forecast:", result["mean"])
print("Forecast shape:", result["mean"].shape)
print("Fitted values shape:", result["fitted"].shape)

assert "mean" in result
assert "fitted" in result
assert result["mean"].shape == (2,)
assert result["fitted"].shape == (8,)

print("Forecast with fitted values works!")

The **stateless** `forecast()` method fits and predicts in one call — no persistent state is stored.

Setting `fitted=True` also returns in-sample fitted values alongside the forecast:
- `"mean"` — future forecasts of shape `(h,)`
- `"fitted"` — in-sample fitted values of shape `(n,)`

## Predict in Sample with Box-Cox

In [ ]:
# Box-Cox: outputs are back on original scale and finite
print("[Test 3] Box-Cox predict_in_sample")
y = jnp.array([1., 2., 4., 8., 16., 32., 64., 128.])  # strictly positive

model = AutoTBATS(
    season_length=2,
    use_boxcox=True,
    use_trend=True,
    use_damped_trend=False,
    use_arma_errors=False
)
model.fit(y)

res = model.predict_in_sample(level=None)

print("Fitted values shape:", res["fitted"].shape)
print("All finite:", bool(jnp.all(jnp.isfinite(res["fitted"]))))
print("All positive:", bool(jnp.all(res["fitted"] > 0)))

lam = model.model_.get("BoxCox_lambda", None)
print(f"Box-Cox lambda: {lam}")
print(f"Fitted (first 5): {res['fitted'][:5]}")

assert "fitted" in res
assert res["fitted"].shape == (y.shape[0],)
assert jnp.all(jnp.isfinite(res["fitted"]))
assert jnp.all(res["fitted"] > 0)

print("Box-Cox in-sample back-transform OK!")

When Box–Cox is enabled:
- Fitted values are automatically back-transformed to the **original scale** (all finite and strictly positive)
- The Box–Cox parameter λ is learned during fitting
- `"fitted"` has the same length as the input series

## Predict with Prediction Intervals

In [ ]:
# Predict with native (Gaussian) prediction intervals
y = jnp.array([10., 12., 13., 15., 17., 20., 22., 25., 27., 30.])

model = AutoTBATS(
    season_length=3,
    use_boxcox=False,
    use_trend=True,
    use_damped_trend=False,
    use_arma_errors=False
)
model.fit(y)

result = model.predict(h=5, level=[80, 95])

print("Forecast:", result["mean"])
print("Lower 80%:", result["lo-80"])
print("Upper 80%:", result["hi-80"])
print("Lower 95%:", result["lo-95"])
print("Upper 95%:", result["hi-95"])

# Verify intervals are consistent: lo <= mean <= hi
for lv in [80, 95]:
    lo = result[f"lo-{lv}"]
    hi = result[f"hi-{lv}"]
    assert jnp.all(lo <= result["mean"] + 1e-12), f"lo-{lv} should be <= mean"
    assert jnp.all(result["mean"] <= hi + 1e-12), f"mean should be <= hi-{lv}"

print("\nPrediction intervals are consistent!")

# Edge Cases & Limitations

TBATS is the go-to model for complex, multi-seasonal time series where simpler ETS or ARIMA models are insufficient.

**Edge Cases:**
1. **Short samples** relative to the seasonal period may cause ARMA or damped-trend components to be disabled during automatic selection
2. **Box–Cox requires strictly positive data** — the transformation is undefined for zero or negative values
3. **Near-zero forecast variance** may produce degenerate intervals (handled defensively)

**Limitations:**
1. **Computationally heavier** than ETS or ARIMA due to the expanded state space
2. **Can overfit** if the seasonal period is mis-specified or the series is too short
3. **Single-step optimization** — like ETS, the optimizer may settle on a local minimum